# AI Interview Simulator

In [1]:
!pip install -q fastapi uvicorn python-multipart pyngrok pypdf langchain langchain-core langchain-community langchain-chroma langchain-text-splitters langchain-google-genai chromadb pydantic

In [6]:
import os
from kaggle_secrets import UserSecretsClient

s = UserSecretsClient()
os.environ["GOOGLE_API_KEY"] = s.get_secret("GOOGLE_API_KEY")
NGROK_TOKEN = s.get_secret("NGROK_TOKEN")
os.environ["API_PASSWORD"] = s.get_secret("API_PASSWORD")
try:
    os.environ["CHAT_MODEL"] = s.get_secret("CHAT_MODEL")
except Exception:
    os.environ["CHAT_MODEL"] = "gemini-flash-lite-latest"   # change if this model is retired
print("Keys loaded. Chat model:", os.environ["CHAT_MODEL"])

Keys loaded. Chat model: gemini-flash-lite-latest


In [7]:
import os
WORKDIR = "/kaggle/working/interview_api"
os.makedirs(WORKDIR + "/data", exist_ok=True)
os.chdir(WORKDIR)
print("Working in", os.getcwd())

Working in /kaggle/working/interview_api


## Project files (written to disk by the cells below)

In [8]:
%%writefile lab.py
"""Core logic: RAG + question chain + evaluation chain + structured output (Gemini).
Requires env var GOOGLE_API_KEY.
"""
import os
import time
import uuid
from functools import lru_cache
from pathlib import Path
from typing import List

from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# ---- Config (change model names here if they stop working) ----
CHAT_MODEL = os.getenv("CHAT_MODEL", "gemini-flash-lite-latest")
EMBED_MODEL = os.getenv("EMBED_MODEL", "models/gemini-embedding-001")
DATA_DIR = Path(__file__).parent / "data"


# ---- Output parser schemas ----
class QuestionList(BaseModel):
    questions: List[str] = Field(description="Interview questions, one per item")


class Evaluation(BaseModel):
    score: int = Field(description="Score from 0 to 10", ge=0, le=10)
    missing_points: List[str] = Field(description="Important points the answer missed")
    suggestions: List[str] = Field(description="Concrete tips to improve the answer")
    feedback: str = Field(description="One short paragraph of overall feedback")


@lru_cache(maxsize=1)
def _llm():
    return ChatGoogleGenerativeAI(model=CHAT_MODEL, temperature=0.3)


@lru_cache(maxsize=1)
def _embeddings():
    return GoogleGenerativeAIEmbeddings(model=EMBED_MODEL)


def _run_chain(prompt: ChatPromptTemplate, schema, inputs: dict, retries: int = 3):
    """prompt -> LLM -> structured output (Pydantic), retrying on temporary errors."""
    chain = prompt | _llm().with_structured_output(schema)
    last_error = None
    for attempt in range(retries):
        try:
            return chain.invoke(inputs)
        except Exception as e:  # rate limit, timeout, parse error
            last_error = e
            time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"Model failed after {retries} attempts: {last_error}")


# ---- RAG ----
def build_retriever(job_description: str):
    """Index the job description + sample interview questions, return a retriever."""
    texts = [job_description]
    for f in DATA_DIR.glob("*.txt"):
        if f.name.startswith("sample_cv") or f.name.startswith("sample_jd"):
            continue
        texts.append(f.read_text(encoding="utf-8"))
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = splitter.create_documents(texts)
    # unique collection per interview so old job descriptions never leak into new ones
    store = Chroma.from_documents(
        chunks, _embeddings(), collection_name=f"jd_{uuid.uuid4().hex[:10]}"
    )
    return store.as_retriever(search_kwargs={"k": 3})


def _context(retriever, query: str) -> str:
    docs = retriever.invoke(query)
    return "\n---\n".join(d.page_content for d in docs)


# ---- Chain 1: generate questions ----
_q_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a professional interviewer. Write {n} interview questions "
               "tailored to this candidate and role. Mix technical and behavioral. "
               "Use the reference material for style and requirements."),
    ("human", "CV:\n{cv}\n\nJob description:\n{jd}\n\nReference material:\n{context}"),
])


def generate_questions(cv: str, jd: str, retriever, n: int = 3) -> List[str]:
    result = _run_chain(_q_prompt, QuestionList,
                        {"cv": cv, "jd": jd, "n": n, "context": _context(retriever, jd)})
    return result.questions[:n]


# ---- Chain 2: evaluate an answer ----
_e_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a strict but fair interviewer. Evaluate the candidate's answer "
               "to the question for the given role. Score 0-10: empty, irrelevant or "
               "'I don't know' answers must score 0-2; vague answers 3-5; solid answers "
               "6-8; excellent, specific answers with examples 9-10."),
    ("human", "Role requirements:\n{context}\n\nQuestion: {question}\n\nAnswer: {answer}"),
])


def evaluate_answer(question: str, answer: str, retriever) -> Evaluation:
    return _run_chain(_e_prompt, Evaluation, {
        "question": question,
        "answer": answer,
        "context": _context(retriever, question),
    })


# ---- Final report (no LLM needed) ----
def final_report(results: list) -> dict:
    scores = [r["evaluation"].score for r in results]
    avg = sum(scores) / len(scores) if scores else 0
    strengths = [r["question"] for r in results if r["evaluation"].score >= 7]
    weaknesses = [r["question"] for r in results if r["evaluation"].score <= 4]
    return {"average": round(avg, 1), "strengths": strengths, "weaknesses": weaknesses}


Overwriting lab.py


In [9]:
%%writefile pdf_utils.py
"""Extract text from an uploaded PDF CV."""
from pypdf import PdfReader

MAX_CHARS = 8000  # keep prompts small (and quota-friendly)


def extract_pdf_text(file) -> str:
    """Return the text of a PDF (path or file-like object). Empty string if none found."""
    reader = PdfReader(file)
    pages = [(page.extract_text() or "") for page in reader.pages]
    text = "\n".join(pages).strip()
    return text[:MAX_CHARS]


Overwriting pdf_utils.py


In [10]:
%%writefile jd_generator.py
"""Generate a job description from a target role (uses the same chain pattern as lab.py)."""
from typing import List

from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

from lab import _run_chain


class JobDescription(BaseModel):
    title: str = Field(description="Job title")
    responsibilities: List[str] = Field(description="3-5 main responsibilities")
    requirements: List[str] = Field(description="5-8 key skills and requirements")
    nice_to_have: List[str] = Field(description="2-3 optional skills")


_jd_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an experienced technical recruiter. Write a realistic, concise "
               "job description for the given role. Be specific about tools, "
               "technologies, and skills."),
    ("human", "Role: {role}"),
])


def generate_job_description(role: str) -> str:
    jd = _run_chain(_jd_prompt, JobDescription, {"role": role.strip()})
    lines = [jd.title, "", "Responsibilities:"]
    lines += [f"- {x}" for x in jd.responsibilities]
    lines += ["", "Requirements:"] + [f"- {x}" for x in jd.requirements]
    if jd.nice_to_have:
        lines += ["", "Nice to have:"] + [f"- {x}" for x in jd.nice_to_have]
    return "\n".join(lines)


# ---- One call: read the CV, pick the best-fit role AND write its job description ----
class RoleAndJD(BaseModel):
    role: str = Field(description="Best-fit target job title for this candidate")
    reason: str = Field(description="One short sentence explaining why this role fits the CV")
    responsibilities: List[str] = Field(description="3-5 main responsibilities for the role")
    requirements: List[str] = Field(description="5-8 key skills and requirements")
    nice_to_have: List[str] = Field(description="2-3 optional skills")


_auto_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an experienced technical recruiter. Read the candidate's CV and "
               "choose the single best-fit target role for them. Then write a realistic, "
               "concise job description for that role, specific about tools and skills."),
    ("human", "CV:\n{cv}"),
])


def suggest_role_and_jd(cv: str):
    """Returns (role, reason, job_description_text) using ONE LLM request."""
    r = _run_chain(_auto_prompt, RoleAndJD, {"cv": cv.strip()[:6000]})
    lines = [r.role, "", "Responsibilities:"]
    lines += [f"- {x}" for x in r.responsibilities]
    lines += ["", "Requirements:"] + [f"- {x}" for x in r.requirements]
    if r.nice_to_have:
        lines += ["", "Nice to have:"] + [f"- {x}" for x in r.nice_to_have]
    return r.role, r.reason, "\n".join(lines)


Overwriting jd_generator.py


In [24]:
%%writefile api.py
"""FastAPI service for the AI Interview Simulator.
Run:  uvicorn api:app --port 8000      Docs:  http://localhost:8000/docs
"""
import hashlib
import hmac
import io
import os
from types import SimpleNamespace
from typing import List

from fastapi import Depends, FastAPI, File, Header, HTTPException, Request, UploadFile
from pydantic import BaseModel, Field

from lab import build_retriever, generate_questions, evaluate_answer, final_report
from jd_generator import suggest_role_and_jd, generate_job_description
from pdf_utils import extract_pdf_text

def require_password(request: Request, authorization: str = Header(default="")):
    if request.url.path == "/health":      # keep health check open
        return
    expected = os.environ.get("API_PASSWORD", "")
    if not expected or not hmac.compare_digest(authorization, f"Bearer {expected}"):
        raise HTTPException(status_code=401, detail="Wrong or missing password.")


app = FastAPI(
    title="AI Interview Simulator API",
    description="RAG + LangChain chains + Pydantic output parsing, powered by Gemini.",
    version="1.0.0",
    dependencies=[Depends(require_password)],
)
# One retriever per job description (building it costs embedding calls, so cache it)
_retrievers = {}


def _retriever(job_description: str):
    key = hashlib.md5(job_description.encode("utf-8")).hexdigest()
    if key not in _retrievers:
        _retrievers[key] = build_retriever(job_description)
    return _retrievers[key]


def _fail(e: Exception):
    print("MODEL ERROR:", repr(e))
    raise HTTPException(status_code=502, detail=f"Model call failed: {e}")

# ---------- request / response schemas ----------
class CVRequest(BaseModel):
    cv: str = Field(min_length=20, description="CV text")


class RoleRequest(BaseModel):
    role: str = Field(min_length=2, examples=["AI/ML Engineer"])


class QuestionsRequest(BaseModel):
    cv: str = Field(min_length=20)
    job_description: str = Field(min_length=20)
    n: int = Field(default=3, ge=1, le=8)


class EvaluateRequest(BaseModel):
    question: str
    answer: str = Field(min_length=1)
    job_description: str = Field(min_length=20)


class ScoredAnswer(BaseModel):
    question: str
    score: int = Field(ge=0, le=10)


class ReportRequest(BaseModel):
    results: List[ScoredAnswer] = Field(min_length=1)


# ---------- endpoints ----------

app = FastAPI()

@app.get("/")
def root():
    return {"message": "API is running. See /docs for endpoints."}
@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/upload-cv", summary="Extract text from a PDF CV (no LLM call)")
async def upload_cv(file: UploadFile = File(...)):
    if not (file.filename or "").lower().endswith(".pdf"):
        raise HTTPException(status_code=400, detail="Please upload a PDF file.")
    try:
        text = extract_pdf_text(io.BytesIO(await file.read()))
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Could not read the PDF: {e}")
    if not text:
        raise HTTPException(status_code=422, detail="No text found (scanned PDF?).")
    return {"cv": text}


@app.post("/suggest-role", summary="Best-fit role + job description from a CV (1 LLM call)")
def suggest_role(req: CVRequest):
    try:
        role, reason, jd = suggest_role_and_jd(req.cv)
    except Exception as e:
        _fail(e)
    return {"role": role, "reason": reason, "job_description": jd}


@app.post("/job-description", summary="Job description from a role title (1 LLM call)")
def job_description(req: RoleRequest):
    try:
        return {"job_description": generate_job_description(req.role)}
    except Exception as e:
        _fail(e)


@app.post("/questions", summary="Generate interview questions (1 LLM call)")
def questions(req: QuestionsRequest):
    try:
        qs = generate_questions(req.cv, req.job_description, _retriever(req.job_description), req.n)
    except Exception as e:
        _fail(e)
    return {"questions": qs}


@app.post("/evaluate", summary="Score one answer (1 LLM call)")
def evaluate(req: EvaluateRequest):
    try:
        ev = evaluate_answer(req.question, req.answer, _retriever(req.job_description))
    except Exception as e:
        _fail(e)
    return ev.model_dump()


@app.post("/report", summary="Final report from scores (no LLM call)")
def report(req: ReportRequest):
    items = [
        {"question": r.question, "evaluation": SimpleNamespace(score=r.score)}
        for r in req.results
    ]
    return final_report(items)


Overwriting api.py


In [25]:
%%writefile data/sample_questions.txt
Tell me about a challenging project you worked on and how you solved the problems you faced.
How do you handle disagreements with a teammate about a technical decision?
Explain a time you had to learn a new technology quickly. What was your approach?
How do you test and debug your code before delivering it?
Describe how you would design a simple REST API for a todo application.
What is the difference between SQL and NoSQL databases, and when would you use each?
Where do you see yourself in two years, and how does this role fit that plan?


Overwriting data/sample_questions.txt


In [26]:
%%writefile data/sample_cv.txt
Python developer with 2 years of experience. Built REST APIs with Django and Flask,
worked with PostgreSQL and Redis, basic Docker knowledge. Final project: a library
management system. Familiar with Git and unit testing with pytest.


Overwriting data/sample_cv.txt


In [27]:
import traceback
os.chdir("/kaggle/working/interview_api")
import sys; sys.path.insert(0, os.getcwd())

from jd_generator import suggest_role_and_jd
try:
    print(suggest_role_and_jd(open("data/sample_cv.txt").read()))
except Exception:
    traceback.print_exc()

('Junior Python Developer', 'The candidate has foundational 2 years of experience building REST APIs using Python, Django, Flask, PostgreSQL, and Redis.', 'Junior Python Developer\n\nResponsibilities:\n- Design and develop scalable RESTful APIs using Python frameworks like Django and Flask\n- Integrate and manage relational databases and caching layers using PostgreSQL and Redis\n- Write clean, maintainable code and perform unit testing using pytest\n- Collaborate with the engineering team using Git for version control and workflow management\n\nRequirements:\n- 2+ years of professional or project experience in Python development\n- Hands-on experience with Django or Flask frameworks\n- Familiarity with relational databases such as PostgreSQL\n- Experience with caching mechanisms like Redis\n- Proficiency in writing unit tests using pytest\n- Basic knowledge of containerization tools like Docker\n- Strong understanding of Git version control\n\nNice to have:\n- Experience with cloud pl

In [28]:
%%writefile data/sample_jd.txt
Backend Engineer. Requirements: Python, REST APIs, PostgreSQL, Docker, testing,
teamwork and clear communication. Nice to have: cloud basics (AWS), CI/CD.


Overwriting data/sample_jd.txt


## Start the API server and the ngrok tunnel

In [33]:
import subprocess, time, requests
from pyngrok import ngrok

# stop anything left from a previous run of this cell
try:
    ngrok.kill()
except Exception:
    pass
try:
    server.terminate()
except Exception:
    pass

server = subprocess.Popen(
    ["uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000"], cwd=WORKDIR
)
time.sleep(10)
print("Local health check:", requests.get("http://127.0.0.1:8000/health").json())

ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(8000)
PUBLIC_URL = tunnel.public_url
print("Public URL:       ", PUBLIC_URL)
print("Interactive docs: ", PUBLIC_URL + "/docs")

INFO:     Started server process [605]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:54894 - "GET /health HTTP/1.1" 200 OK
Local health check: {'status': 'ok'}
Public URL:        https://giant-enduring-record.ngrok-free.dev
Interactive docs:  https://giant-enduring-record.ngrok-free.dev/docs
INFO:     156.192.140.87:0 - "POST /suggest-role HTTP/1.1" 200 OK


## Stop everything when you're done

In [ ]:
ngrok.kill()
server.terminate()
print("Stopped.")